# 04. FDS V6 Model Serving · 교육용 MVP
## 저장된 모델 + 최종 설정 → Inference → Streamlit → Docker

이 Notebook에서는 **모델 학습이나 Threshold 튜닝을 하지 않습니다.** 03에서 확정하고 저장한 Artifact를 그대로 사용합니다.

```text
03 Model Development
├─ models/fds_model.joblib
├─ models/fds_model_metadata.json
├─ models/fds_hybrid_metadata.json
├─ models/fds_rule_config.json
├─ models/fds_model.sha256
├─ output/fds_realtime_demo.csv.gz
└─ requirements.streamlit.txt
        ↓
04 Model Serving
        ↓
Artifact 무결성 / Runtime 버전 확인
        ↓
1건 Inference / Batch Inference
        ↓
Rule + ML → Hybrid Score
        ↓
Protection Action
        ↓
Streamlit / Docker
```

> 이 MVP는 **Test에서 이미 Feature Engineering이 끝난 대표 Event**를 서비스처럼 한 건씩 재생합니다. `fraud_label`, `fraud_type`은 교육용 정답 확인에만 사용하며 모델 입력이나 의사결정에는 사용하지 않습니다.
>
> 실제 운영 FDS에서는 신규 원시 거래가 들어올 때 최근 10분/1시간/30일 이력, 신규기기·신규수취인 등의 Feature와 Rule Score를 실시간으로 계산하는 Online Feature Engineering 계층이 추가로 필요합니다.


## 1. Serving Artifact 로드

03에서 저장한 모델과 Metadata를 읽습니다.

04에서는 다음을 **절대 재선택하지 않습니다.**

- ML 모델
- ML Threshold
- Rule Threshold
- ML/Rule Weight
- Hybrid Threshold
- Action Threshold

In [1]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import hashlib
import json
import platform
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn

warnings.filterwarnings('ignore')
PROJECT_DIR = Path.cwd(); OUTPUT_DIR = PROJECT_DIR / 'output'; MODEL_DIR = PROJECT_DIR / 'models'
MODEL_PATH = MODEL_DIR / 'fds_model.joblib'; ML_META_PATH = MODEL_DIR / 'fds_model_metadata.json'; HYBRID_META_PATH = MODEL_DIR / 'fds_hybrid_metadata.json'; RULE_CONFIG_PATH = MODEL_DIR / 'fds_rule_config.json'; MODEL_HASH_PATH = MODEL_DIR / 'fds_model.sha256'; DATA_PATH = OUTPUT_DIR / 'fds_realtime_demo.csv.gz'
required_paths = [MODEL_PATH, ML_META_PATH, HYBRID_META_PATH, RULE_CONFIG_PATH, MODEL_HASH_PATH, DATA_PATH]
for path in required_paths:
    if not path.exists(): raise FileNotFoundError(f'필수 Serving Artifact가 없습니다: {path}')

expected_hash = MODEL_HASH_PATH.read_text(encoding='utf-8').strip(); actual_hash = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
if expected_hash != actual_hash: raise RuntimeError('fds_model.joblib SHA256이 저장된 값과 다릅니다. 모델 파일이 바뀌었는지 확인하세요.')

model = joblib.load(MODEL_PATH)
with open(ML_META_PATH, 'r', encoding='utf-8') as f: ml_meta = json.load(f)
with open(HYBRID_META_PATH, 'r', encoding='utf-8') as f: hybrid_meta = json.load(f)
with open(RULE_CONFIG_PATH, 'r', encoding='utf-8') as f: rule_meta = json.load(f)
events = pd.read_csv(DATA_PATH, low_memory=False); events['event_at'] = pd.to_datetime(events['event_at'], errors='coerce')

MODEL_FEATURES = ml_meta['model_features']; NUMERIC_FEATURES = ml_meta.get('numeric_features', []); CATEGORICAL_FEATURES = ml_meta.get('categorical_features', [])
ML_WEIGHT = float(hybrid_meta['ml_weight']); RULE_WEIGHT = float(hybrid_meta['rule_weight']); HYBRID_THRESHOLD = float(hybrid_meta['hybrid_threshold']); FRAUD_ACTION_THRESHOLDS = hybrid_meta['fraud_action_thresholds']
missing_features = [c for c in MODEL_FEATURES if c not in events.columns]
if missing_features: raise RuntimeError(f'Demo Dataset Feature 누락: {missing_features}')

current_versions = {'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'scikit_learn': sklearn.__version__, 'joblib': joblib.__version__}
for package, key in [('xgboost','xgboost'), ('lightgbm','lightgbm')]:
    try: current_versions[key] = version(package)
    except PackageNotFoundError: current_versions[key] = None
expected_versions = ml_meta.get('runtime_versions', {})
version_check = pd.DataFrame([{'package': k, 'training': expected_versions.get(k), 'serving': current_versions.get(k), 'status': 'OK' if (k == 'python' and str(expected_versions.get(k,'')).split('.')[:2] == str(current_versions.get(k,'')).split('.')[:2]) or (k != 'python' and expected_versions.get(k) == current_versions.get(k)) else 'CHECK'} for k in sorted(set(expected_versions) | set(current_versions))])

print('Model:', ml_meta['model_name']); print('Model SHA256: OK'); print('ML_WEIGHT:', ML_WEIGHT); print('RULE_WEIGHT:', RULE_WEIGHT); print('HYBRID_THRESHOLD:', HYBRID_THRESHOLD); print('Demo rows:', f'{len(events):,}')
display(version_check)
if (version_check['status'] == 'CHECK').any(): print('주의: 학습 환경과 Serving 환경의 버전 차이가 있습니다. 03이 생성한 requirements.streamlit.txt 사용을 권장합니다.')


Model: LightGBM_l31
Model SHA256: OK
ML_WEIGHT: 0.9000000000000004
RULE_WEIGHT: 0.09999999999999964
HYBRID_THRESHOLD: 0.920786976511933
Demo rows: 117


,package,training,serving,status
0,joblib,1.5.1,1.5.1,OK
1,lightgbm,4.7.0,4.7.0,OK
2,numpy,2.3.2,2.3.2,OK
3,pandas,3.0.0,3.0.0,OK
4,python,3.11.16,3.11.16,OK
5,scikit_learn,1.7.2,1.7.2,OK
6,xgboost,3.2.0,3.2.0,OK


## 2. Serving 함수

실제 서비스의 추론 흐름은 다음과 같습니다.

```text
Feature 1건
→ ML Pipeline predict_proba()
→ ML Risk Score
+
Rule Score
→ Hybrid Score
→ Alert 여부
→ Protection Action
```

여기서는 03에서 만든 Demo Feature를 사용합니다.

In [2]:
def make_model_input(row):
    X = pd.DataFrame([{feature: row.get(feature, np.nan) for feature in MODEL_FEATURES}])
    for col in NUMERIC_FEATURES:
        if col in X.columns: X[col] = pd.to_numeric(X[col], errors='coerce').astype('float64')
    for col in CATEGORICAL_FEATURES:
        if col in X.columns:
            s = X[col].astype('object'); X[col] = s.where(pd.notna(s), np.nan)
    return X

def decide_action(score):
    if score >= FRAUD_ACTION_THRESHOLDS['block']: return 'BLOCK_AND_REVIEW'
    if score >= FRAUD_ACTION_THRESHOLDS['temporary_hold']: return 'TEMP_HOLD'
    if score >= FRAUD_ACTION_THRESHOLDS['step_up']: return 'STEP_UP_AUTH'
    return 'PASS'

def safe_number(row, key, default=0.0): return float(pd.to_numeric(pd.Series([row.get(key, default)]), errors='coerce').fillna(default).iloc[0])

def predict_event(row):
    started = time.perf_counter(); model_input = make_model_input(row); ml_risk_score = float(model.predict_proba(model_input)[:, 1][0]); rule_score_100 = safe_number(row, 'rule_score', 0); rule_score = float(np.clip(rule_score_100 / 100, 0, 1)); hybrid_score = ML_WEIGHT * ml_risk_score + RULE_WEIGHT * rule_score
    return {'event_uid': str(row.get('event_uid', '')), 'event_at': row.get('event_at'), 'source_table': str(row.get('source_table', '')), 'event_subtype': str(row.get('event_subtype', '')), 'amount_abs': safe_number(row, 'amount_abs', 0), 'rule_score': rule_score_100, 'ml_risk_score': ml_risk_score, 'hybrid_score': float(hybrid_score), 'alert': int(hybrid_score >= HYBRID_THRESHOLD), 'action': decide_action(hybrid_score), 'latency_ms': (time.perf_counter() - started) * 1000}


## 3. 거래 1건 Inference

이 셀에서는 Demo의 첫 번째 Event 1건을 실제 Serving 함수에 넣습니다.

정답 컬럼은 결과 계산에 사용하지 않습니다.

In [3]:
sample_row = events.iloc[0]
result = predict_event(sample_row)

display(pd.DataFrame([result]))
print('교육용 실제 정답:', sample_row.get('fraud_label', 'N/A'), '/', sample_row.get('fraud_type', 'N/A'))


,event_uid,event_at,source_table,event_subtype,amount_abs,rule_score,ml_risk_score,hybrid_score,alert,action,latency_ms
0,card_payments:2834908,2026-07-15 12:26:00,card_payments,ECOMMERCE,897897.9,32.0,0.999904,0.931913,1,TEMP_HOLD,39.449


교육용 실제 정답: 1 / CARD_NOT_PRESENT


## 4. Batch Inference

실서비스에서는 Kafka/API 등에서 Event가 연속으로 들어오지만,
교육용에서는 Demo DataFrame의 여러 행을 순서대로 처리합니다.

In [4]:
BATCH_SIZE = min(20, len(events))

batch_results = [predict_event(row) for _, row in events.head(BATCH_SIZE).iterrows()]
batch_result_df = pd.DataFrame(batch_results)

display(batch_result_df)
print('평균 추론시간:', f'{batch_result_df["latency_ms"].mean():.2f} ms')


,event_uid,event_at,source_table,event_subtype,amount_abs,rule_score,ml_risk_score,hybrid_score,alert,action,latency_ms
0,card_payments:2834908,2026-07-15 12:26:00,card_payments,ECOMMERCE,897897.90,32.0,0.999904,0.931913,1,TEMP_HOLD,34.8502
1,card_payments:17385083,2026-07-15 14:52:00,card_payments,POS,707807.67,20.0,0.930604,0.857543,0,STEP_UP_AUTH,23.9491
2,card_payments:22476499,2026-07-15 17:18:00,card_payments,ECOMMERCE,147294.22,12.0,0.996108,0.908497,0,STEP_UP_AUTH,23.6038
3,card_payments:14567045,2026-07-15 18:43:00,card_payments,ECOMMERCE,127143.67,32.0,0.995389,0.927850,1,TEMP_HOLD,23.2461
4,card_payments:7522032,2026-07-16 10:03:00,card_payments,POS,256532.50,10.0,0.928180,0.845362,0,STEP_UP_AUTH,24.0414
5,card_payments:14208222,2026-07-16 10:19:00,card_payments,POS,519533.66,10.0,0.990711,0.901640,0,STEP_UP_AUTH,22.4409
6,card_payments:19504768,2026-07-16 14:33:00,card_payments,POS,359295.49,20.0,0.982746,0.904472,0,STEP_UP_AUTH,22.7497
7,card_payments:17616807,2026-07-16 15:51:00,card_payments,ECOMMERCE,101992.62,32.0,0.966505,0.901854,0,STEP_UP_AUTH,23.2421
8,card_payments:16009330,2026-07-16 15:56:00,card_payments,POS,227272.80,20.0,0.733491,0.680142,0,STEP_UP_AUTH,22.7873
9,card_payments:13357703,2026-07-16 16:11:00,card_payments,POS,82174.09,20.0,0.546183,0.511565,0,PASS,22.9606


평균 추론시간: 23.32 ms


## 5. Serving과 성능평가의 차이

04에서는 모델을 다시 학습하거나 Validation/Test를 보고 설정을 바꾸지 않습니다.

```text
X 모델 재학습
X Validation으로 Weight 재선택
X Threshold 재조정
X Test 결과를 보고 설정 수정
```

Serving은 이미 확정된 모델과 설정으로 **새 Event와 같은 구조의 Feature에 동일한 추론 로직을 적용**하는 단계입니다.

현재 `fds_realtime_demo.csv.gz`는 03의 최종 Test에서 대표 사례만 골라 만든 **교육용 재생 데이터**입니다. 따라서 화면에서 한 건씩 처리되지만 원시 거래가 실시간으로 들어와 Feature를 즉석 생성하는 구조는 아닙니다.

### 실제 운영으로 확장할 때 추가되는 계층

```text
신규 원시 거래
→ Online Feature Engineering / Feature Store
→ 최근 10분·1시간·24시간·30일 이력 Feature 계산
→ 신규기기·신규수취인·해외IP 등 Rule Feature 계산
→ Rule Score
→ 저장된 ML Pipeline predict_proba()
→ Hybrid Score
→ PASS / STEP_UP_AUTH / TEMP_HOLD / BLOCK_AND_REVIEW
```

Demo의 `fraud_label`과 `fraud_type`은 학생이 예측과 정답을 비교하기 위한 교육용 정보이며 의사결정에는 사용하지 않습니다.


## 6. Streamlit 교육용 MVP 실행

프로젝트 루트에는 아래 **정확한 파일명**을 사용합니다.

```text
app_realtime_fds.py
requirements.streamlit.txt
models/...
output/...
```

실행:

```bash
pip install -r requirements.streamlit.txt
streamlit run app_realtime_fds.py
```

브라우저: `http://localhost:8501`

Streamlit 앱은 이 Notebook과 동일하게 저장된 Model/Metadata를 읽기만 하며 Threshold를 재튜닝하지 않습니다. 시작할 때 모델 SHA256과 학습/Serving 라이브러리 버전도 확인합니다.


## 7. Docker Serving

03을 마지막까지 실행하면 모델 Artifact와 `requirements.streamlit.txt`가 준비됩니다.

```text
app_realtime_fds.py
Dockerfile.streamlit
docker-compose.local.yml
requirements.streamlit.txt

models/
├─ fds_model.joblib
├─ fds_model_metadata.json
├─ fds_hybrid_metadata.json
├─ fds_rule_config.json
└─ fds_model.sha256

output/
└─ fds_realtime_demo.csv.gz
```

> 파일명 뒤에 `(5)`, `(1)` 같은 다운로드 중복번호가 붙어 있으면 제거하고 위 이름으로 맞춥니다. 이번 수정본은 이미 위 표준 이름으로 제공합니다.

Build / Run:

```bash
docker compose -f docker-compose.local.yml build --no-cache
docker compose -f docker-compose.local.yml up -d
docker compose -f docker-compose.local.yml ps
```

로그 확인:

```bash
docker compose -f docker-compose.local.yml logs -f fds-mvp
```

접속: `http://localhost:8501`

### 이 Docker가 구현하는 범위

현재 컨테이너는 **저장된 Feature 기반 교육용 Serving MVP**를 실행합니다. 실제 운영 환경에서는 DB/Kafka/API 등으로 신규 거래를 받고 Online Feature Engineering 계층을 별도 서비스로 구성한 뒤 이 모델 Serving 계층에 전달하는 구조가 필요합니다.
